# Lektion 11 - Agent-til-Agent (A2A) Protokol


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hvad er A2A-protokollen?

**Agent-til-Agent (A2A) protokollen** er en åben standard, der gør det muligt for AI-agenter at kommunikere,
opdage hinanden og samarbejde – selv når de er bygget på forskellige rammer eller hostet
af forskellige tjenester.

Nøglebegreber:

- **Opdagelse** – Agenter publicerer et *Agentkort*, der beskriver deres kapaciteter, hvilket gør det
  nemt for andre agenter (eller orkestratorer) at finde den rette specialist til en opgave.
- **Beskedudveksling** – Agenter udveksler strukturerede beskeder gennem en fælles protokol, så en
  anmodning fra en agent kan forstås og opfyldes af en anden uanset dens interne
  implementering.
- **Opgavens livscyklus** – A2A definerer tilstande som *indsendt*, *arbejder på*, *fuldført* og
  *mislykkedes*, hvilket giver orkestratoren fuld indsigt i, hvordan en delegeret opgave skrider frem.

I denne lektion simulerer vi A2A-stil samarbejde ved at forbinde tre specialiserede rejseagenter
i en arbejdsgang, hvor hver agent bidrager med sin ekspertise og sender resultater videre til den næste.


## Oprettelse af specialiserede rejsebureauer


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Multi-Agent Collaboration via Workflow

We connect the three agents into a sequential workflow that mirrors A2A message passing:

1. **CurrencyExchangeAgent** receives the user request and produces currency guidance.
2. **ActivityPlannerAgent** receives the enriched context and adds activity recommendations.
3. **TravelManagerAgent** synthesizes both inputs into a final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Forståelse af A2A i Produktion

I et produktionsmiljø åbner A2A-protokollen op for kraftfulde tvær-service scenarier:

| Funktionalitet | Beskrivelse |
|---|---|
| **Tvær-rammeværks interoperabilitet** | En agent bygget med ét framework kan delegere opgaver til en agent bygget med ethvert andet A2A-kompatibelt framework, hvilket muliggør ægte tvær-organisation interoperabilitet. |
| **Servicegrænser** | Agenter kan bo i separate mikroservices, cloud-regioner eller endda forskellige organisationer, mens de stadig samarbejder problemfrit. |
| **Dynamisk opdagelse** | En orkestrator kan forespørge et Agent Card-register under kørslen for at finde den bedst egnede specialist til en given delopgave. |
| **Streaming & push-notifikationer** | A2A understøtter Server-Sent Events (SSE) for realtids fremskridtsovervågning og push-notifikationer for langvarige opgaver. |

Den workflow, vi byggede ovenfor, er en forenklet, in-process version af dette mønster. I en rigtig
implementering ville hver agent eksponere en HTTP-endpoint, publicere et Agent Card og kommunikere
via A2A JSON-RPC protokollen.


## Resumé

I denne lektion lærte du:

1. **Hvad A2A-protokollen er** — en åben standard for agent-til-agent opdagelse, beskedudveksling,
   og opgavestyring.
2. **Hvordan man opretter specialiserede agenter** — en valutaomvekslingsagent, en aktivitetsplanlægningsagent,
   og en rejseadministrator som orkestrator.
3. **Hvordan man forbinder agenter i en arbejdsgang** — ved at bruge `WorkflowBuilder` til at modellere sekventiel
   beskedudveksling mellem agenter.
4. **Hvordan A2A fungerer i produktion** — muliggør tværgående samarbejde på tværs af rammeværk og tjenester
   med dynamisk opdagelse og streamingopdateringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
